# Stock Price Prediction Model
## Training on News Sentiment Data (Multi-Stock, Multi-Task Learning)

This notebook trains a deep learning model to predict 1-day, 2-day, and 3-day stock returns based on news headlines from 50 different stocks using:
- **BERT** for news headline encoding
- **Multi-task learning** for predicting ret_1d, ret_2d, ret_3d simultaneously
- **Transfer learning** from pre-trained BERT model

**Dataset**: 50 aligned CSVs from moneycontrol + yfinance, with columns:
- `headline`: News headline text
- `ret_1d`, `ret_2d`, `ret_3d`: Target returns

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
print(f"✓ GPU available: {torch.cuda.is_available()}")

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

ImportError: cannot import name 'AdamW' from 'transformers' (c:\Users\aayus\Dev\Stocks Senti\MyEnv\Lib\site-packages\transformers\__init__.py)

## 2. Load and Explore the Dataset

In [ ]:
# Load all aligned CSV files
aligned_dir = Path('Data/processed/aligned')
csv_files = list(aligned_dir.glob('*_aligned.csv'))

print(f"Found {len(csv_files)} aligned CSV file(s)")
print(f"Files: {[f.stem.replace('_aligned', '') for f in csv_files]}\n")

# Load and combine all data
all_data = []
for csv_file in csv_files:
    stock_name = csv_file.stem.replace('_aligned', '')
    df = pd.read_csv(csv_file)
    df['stock'] = stock_name
    all_data.append(df)
    print(f"  {stock_name}: {len(df)} records")

# Combine all dataframes
combined_df = pd.concat(all_data, ignore_index=True)
combined_df = combined_df.drop_duplicates()

print(f"\nTotal combined records: {len(combined_df)}")
print(f"\nDataset shape: {combined_df.shape}")
print(f"\nColumn names:\n{combined_df.columns.tolist()}")
print(f"\nFirst few rows:")
combined_df.head()

In [ ]:
# Data exploration
print("="*60)
print("DATA EXPLORATION")
print("="*60)

# Check for missing values
print("\nMissing values:")
print(combined_df.isnull().sum())

# Check data types
print("\nData types:")
print(combined_df.dtypes)

# Statistics for target variables
print("\nTarget variables statistics (ret_1d, ret_2d, ret_3d):")
print(combined_df[['ret_1d', 'ret_2d', 'ret_3d']].describe())

# Check for any NaN or Inf in targets
print(f"\nRows with NaN in ret_1d: {combined_df['ret_1d'].isna().sum()}")
print(f"Rows with NaN in ret_2d: {combined_df['ret_2d'].isna().sum()}")
print(f"Rows with NaN in ret_3d: {combined_df['ret_3d'].isna().sum()}")

# Sample headlines
print("\nSample headlines:")
for i, headline in enumerate(combined_df['headline'].head(3)):
    print(f"  {i+1}. {headline[:80]}...")

# Distribution of returns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
combined_df['ret_1d'].hist(ax=axes[0], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of 1-Day Returns')
axes[0].set_xlabel('Return')

combined_df['ret_2d'].hist(ax=axes[1], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of 2-Day Returns')
axes[1].set_xlabel('Return')

combined_df['ret_3d'].hist(ax=axes[2], bins=50, edgecolor='black', alpha=0.7)
axes[2].set_title('Distribution of 3-Day Returns')
axes[2].set_xlabel('Return')

plt.tight_layout()
plt.show()

print("\n" + "="*60)

## 3. Data Preprocessing and Feature Engineering

In [ ]:
# Data cleaning
print("Data Preprocessing...")

# Remove rows with NaN in critical columns
df_clean = combined_df.dropna(subset=['headline', 'ret_1d', 'ret_2d', 'ret_3d']).copy()
print(f"✓ Removed rows with NaN values. Remaining: {len(df_clean)}")

# Remove duplicate headlines
df_clean = df_clean.drop_duplicates(subset=['headline', 'event_date'])
print(f"✓ Removed duplicates. Remaining: {len(df_clean)}")

# Check for any inf values
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()
print(f"✓ Removed inf values. Remaining: {len(df_clean)}")

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f"\nFinal dataset shape: {df_clean.shape}")
print(f"Target variables - Mean:")
print(f"  ret_1d: {df_clean['ret_1d'].mean():.6f}")
print(f"  ret_2d: {df_clean['ret_2d'].mean():.6f}")
print(f"  ret_3d: {df_clean['ret_3d'].mean():.6f}")

print(f"\nTarget variables - Std:")
print(f"  ret_1d: {df_clean['ret_1d'].std():.6f}")
print(f"  ret_2d: {df_clean['ret_2d'].std():.6f}")
print(f"  ret_3d: {df_clean['ret_3d'].std():.6f}")

In [ ]:
# Feature engineering: Basic text features
print("\nFeature Engineering...")

# Headline length (number of words)
df_clean['headline_length'] = df_clean['headline'].str.split().str.len()
print(f"✓ Added headline_length feature")

# Standardize returns to help training
scaler_1d = StandardScaler()
scaler_2d = StandardScaler()
scaler_3d = StandardScaler()

df_clean['ret_1d_scaled'] = scaler_1d.fit_transform(df_clean[['ret_1d']])
df_clean['ret_2d_scaled'] = scaler_2d.fit_transform(df_clean[['ret_2d']])
df_clean['ret_3d_scaled'] = scaler_3d.fit_transform(df_clean[['ret_3d']])

print(f"✓ Scaled target variables")
print(f"\nFeatures ready for model training!")

## 4. Prepare Training and Testing Sets

In [ ]:
# Split data without shuffling to preserve temporal order (avoid leakage)
train_size = 0.70
val_size = 0.15
test_size = 0.15

n_train = int(len(df_clean) * train_size)
n_val = int(len(df_clean) * val_size)

train_df = df_clean[:n_train].reset_index(drop=True)
val_df = df_clean[n_train:n_train+n_val].reset_index(drop=True)
test_df = df_clean[n_train+n_val:].reset_index(drop=True)

print("Train-Test Split (time-ordered):")
print(f"  Training set: {len(train_df)} samples ({100*train_size:.1f}%)")
print(f"  Validation set: {len(val_df)} samples ({100*val_size:.1f}%)")
print(f"  Test set: {len(test_df)} samples ({100*test_size:.1f}%)")

print(f"\nSplit date ranges maintained for temporal integrity")

In [ ]:
# Custom PyTorch Dataset class
class StockNewsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.headlines = dataframe['headline'].reset_index(drop=True)
        self.ret_1d = dataframe['ret_1d'].reset_index(drop=True)
        self.ret_2d = dataframe['ret_2d'].reset_index(drop=True)
        self.ret_3d = dataframe['ret_3d'].reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.headlines)
    
    def __getitem__(self, idx):
        headline = str(self.headlines.iloc[idx])
        
        # Tokenize the headline
        encoding = self.tokenizer(
            headline,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'ret_1d': torch.tensor(self.ret_1d.iloc[idx], dtype=torch.float32),
            'ret_2d': torch.tensor(self.ret_2d.iloc[idx], dtype=torch.float32),
            'ret_3d': torch.tensor(self.ret_3d.iloc[idx], dtype=torch.float32),
        }

print("✓ StockNewsDataset class created")

# Initialize tokenizer
print("\nInitializing BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("✓ Tokenizer initialized")

# Create PyTorch datasets
train_dataset = StockNewsDataset(train_df, tokenizer)
val_dataset = StockNewsDataset(val_df, tokenizer)
test_dataset = StockNewsDataset(test_df, tokenizer)

print(f"✓ Created 3 PyTorch datasets")

# Create data loaders
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"✓ Created data loaders with batch size: {batch_size}")

## 5. Build the Neural Network Model

In [ ]:
# Multi-Task Learning Model Architecture
class MultiTaskStockPredictor(nn.Module):
    """BERT-based model with multi-task learning heads for predicting ret_1d, ret_2d, ret_3d"""
    
    def __init__(self, model_name='bert-base-uncased', hidden_size=768, dropout=0.1):
        super(MultiTaskStockPredictor, self).__init__()
        
        # Load pre-trained BERT
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        
        # Shared feature extraction layers
        self.shared_dense = nn.Linear(hidden_size, 256)
        self.shared_activation = nn.ReLU()
        self.shared_dropout = nn.Dropout(dropout)
        
        # Task-specific heads
        # Head for 1-day return prediction
        self.head_1d = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
        
        # Head for 2-day return prediction
        self.head_2d = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
        
        # Head for 3-day return prediction
        self.head_3d = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    
    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use [CLS] token (first token) representation
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        
        # Shared feature extraction
        shared_features = self.shared_dense(cls_output)
        shared_features = self.shared_activation(shared_features)
        shared_features = self.shared_dropout(shared_features)
        
        # Task-specific predictions
        pred_1d = self.head_1d(shared_features)
        pred_2d = self.head_2d(shared_features)
        pred_3d = self.head_3d(shared_features)
        
        return pred_1d, pred_2d, pred_3d

print("✓ MultiTaskStockPredictor model defined")

# Initialize model
print("\nInitializing model...")
model = MultiTaskStockPredictor('bert-base-uncased')
model.to(device)
print(f"✓ Model initialized on {device}")

# Print model summary
print(f"\nModel Summary:")
print(f"  Parameters: BERT (base) + Multi-task heads")
print(f"  BERT hidden size: 768")
print(f"  Shared layers: 256 units")
print(f"  Task-specific layers: 128 → 1 units each")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total trainable parameters: {total_params:,}")

## 6. Train the Model

In [ ]:
# Setup training components
criterion = nn.MSELoss()  # For regression tasks
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training configuration
num_epochs = 10
patience = 3
best_val_loss = float('inf')
patience_counter = 0

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_loss_1d': [],
    'train_loss_2d': [],
    'train_loss_3d': [],
    'val_loss_1d': [],
    'val_loss_2d': [],
    'val_loss_3d': []
}

print("="*70)
print("TRAINING MODEL")
print("="*70)
print(f"Epochs: {num_epochs}")
print(f"Learning rate: 2e-5")
print(f"Optimizer: AdamW")
print(f"Loss function: MSE (regression)")
print(f"Early stopping patience: {patience}")
print("="*70 + "\n")

# Training loop
for epoch in range(num_epochs):
    # ========== TRAINING ==========
    model.train()
    train_loss = 0
    train_loss_1d = 0
    train_loss_2d = 0
    train_loss_3d = 0
    
    for batch_idx, batch in enumerate(train_dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ret_1d = batch['ret_1d'].to(device)
        ret_2d = batch['ret_2d'].to(device)
        ret_3d = batch['ret_3d'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        pred_1d, pred_2d, pred_3d = model(input_ids, attention_mask)
        
        # Calculate individual task losses
        loss_1d = criterion(pred_1d.squeeze(-1), ret_1d)
        loss_2d = criterion(pred_2d.squeeze(-1), ret_2d)
        loss_3d = criterion(pred_3d.squeeze(-1), ret_3d)
        
        # Total loss (equal weight for all tasks)
        loss = loss_1d + loss_2d + loss_3d
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_loss_1d += loss_1d.item()
        train_loss_2d += loss_2d.item()
        train_loss_3d += loss_3d.item()
    
    avg_train_loss = train_loss / len(train_dataloader)
    avg_train_loss_1d = train_loss_1d / len(train_dataloader)
    avg_train_loss_2d = train_loss_2d / len(train_dataloader)
    avg_train_loss_3d = train_loss_3d / len(train_dataloader)
    
    # ========== VALIDATION ==========
    model.eval()
    val_loss = 0
    val_loss_1d = 0
    val_loss_2d = 0
    val_loss_3d = 0
    
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ret_1d = batch['ret_1d'].to(device)
            ret_2d = batch['ret_2d'].to(device)
            ret_3d = batch['ret_3d'].to(device)
            
            pred_1d, pred_2d, pred_3d = model(input_ids, attention_mask)
            
            loss_1d = criterion(pred_1d.squeeze(-1), ret_1d)
            loss_2d = criterion(pred_2d.squeeze(-1), ret_2d)
            loss_3d = criterion(pred_3d.squeeze(-1), ret_3d)
            
            loss = loss_1d + loss_2d + loss_3d
            
            val_loss += loss.item()
            val_loss_1d += loss_1d.item()
            val_loss_2d += loss_2d.item()
            val_loss_3d += loss_3d.item()
    
    avg_val_loss = val_loss / len(val_dataloader)
    avg_val_loss_1d = val_loss_1d / len(val_dataloader)
    avg_val_loss_2d = val_loss_2d / len(val_dataloader)
    avg_val_loss_3d = val_loss_3d / len(val_dataloader)
    
    # Store history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_loss_1d'].append(avg_train_loss_1d)
    history['train_loss_2d'].append(avg_train_loss_2d)
    history['train_loss_3d'].append(avg_train_loss_3d)
    history['val_loss_1d'].append(avg_val_loss_1d)
    history['val_loss_2d'].append(avg_val_loss_2d)
    history['val_loss_3d'].append(avg_val_loss_3d)
    
    # Print progress
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"    1D: {avg_train_loss_1d:.4f} → {avg_val_loss_1d:.4f} | " +
          f"2D: {avg_train_loss_2d:.4f} → {avg_val_loss_2d:.4f} | " +
          f"3D: {avg_train_loss_3d:.4f} → {avg_val_loss_3d:.4f}")
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model_weights.pth')
        print(f"  ✓ Best model saved (Val Loss: {avg_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n✓ Early stopping triggered after {epoch+1} epochs")
            break
    
    print()

# Load best model
print("Loading best model weights...")
model.load_state_dict(torch.load('best_model_weights.pth'))
print("✓ Best model loaded\n")

In [ ]:
# Visualize training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall loss
axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].set_title('Overall Training Progress')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 1-Day Returns Loss
axes[0, 1].plot(history['train_loss_1d'], label='Train Loss (1d)', marker='o', linewidth=2)
axes[0, 1].plot(history['val_loss_1d'], label='Val Loss (1d)', marker='s', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MSE Loss')
axes[0, 1].set_title('1-Day Returns Prediction')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 2-Day Returns Loss
axes[1, 0].plot(history['train_loss_2d'], label='Train Loss (2d)', marker='o', linewidth=2)
axes[1, 0].plot(history['val_loss_2d'], label='Val Loss (2d)', marker='s', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('MSE Loss')
axes[1, 0].set_title('2-Day Returns Prediction')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 3-Day Returns Loss
axes[1, 1].plot(history['train_loss_3d'], label='Train Loss (3d)', marker='o', linewidth=2)
axes[1, 1].plot(history['val_loss_3d'], label='Val Loss (3d)', marker='s', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('MSE Loss')
axes[1, 1].set_title('3-Day Returns Prediction')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training history visualized")

## 7. Evaluate Model Performance

In [ ]:
# Function to evaluate on a dataset
def evaluate_model(model, dataloader, device, set_name=""):
    model.eval()
    predictions = {'1d': [], '2d': [], '3d': []}
    actuals = {'1d': [], '2d': [], '3d': []}
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ret_1d = batch['ret_1d']
            ret_2d = batch['ret_2d']
            ret_3d = batch['ret_3d']
            
            pred_1d, pred_2d, pred_3d = model(input_ids, attention_mask)
            
            predictions['1d'].extend(pred_1d.squeeze(-1).cpu().numpy())
            predictions['2d'].extend(pred_2d.squeeze(-1).cpu().numpy())
            predictions['3d'].extend(pred_3d.squeeze(-1).cpu().numpy())
            
            actuals['1d'].extend(ret_1d.numpy())
            actuals['2d'].extend(ret_2d.numpy())
            actuals['3d'].extend(ret_3d.numpy())
    
    # Calculate metrics
    metrics = {}
    for horizon in ['1d', '2d', '3d']:
        y_true = np.array(actuals[horizon])
        y_pred = np.array(predictions[horizon])
        
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        
        metrics[f'ret_{horizon}'] = {
            'MSE': mse,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        }
    
    return metrics, predictions, actuals

print("Evaluating on Test Set...")
print("="*70)

test_metrics, test_preds, test_actuals = evaluate_model(model, test_dataloader, device, "Test")

# Print results
print("\nTEST SET RESULTS:")
print("-"*70)
for horizon, metric_dict in test_metrics.items():
    print(f"\n{horizon.upper()}:")
    for metric_name, metric_value in metric_dict.items():
        print(f"  {metric_name:6s}: {metric_value:.6f}")

print("\n" + "="*70)

In [ ]:
# Visualize predictions vs actuals
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

horizons = ['1d', '2d', '3d']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, horizon in enumerate(horizons):
    # Scatter plot: Predicted vs Actual
    ax = axes[0, idx]
    y_true = np.array(test_actuals[horizon])
    y_pred = np.array(test_preds[horizon])
    
    ax.scatter(y_true, y_pred, alpha=0.5, s=20, color=colors[idx])
    
    # Add diagonal reference line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    ax.set_xlabel('Actual Returns')
    ax.set_ylabel('Predicted Returns')
    ax.set_title(f'{horizon.upper()} Returns: Predicted vs Actual')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Residuals: Error distribution
    ax_res = axes[1, idx]
    errors = y_pred - y_true
    ax_res.hist(errors, bins=40, edgecolor='black', alpha=0.7, color=colors[idx])
    ax_res.axvline(x=0, color='r', linestyle='--', linewidth=2)
    ax_res.set_xlabel('Prediction Error')
    ax_res.set_ylabel('Frequency')
    ax_res.set_title(f'{horizon.upper()} Error Distribution')
    ax_res.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction accuracy visualized")

## 8. Make Predictions on New Data

In [ ]:
# Function to make predictions on single headlines
def predict_single_headline(headline, model, tokenizer, device):
    """Predict returns for a single news headline"""
    model.eval()
    
    # Tokenize
    inputs = tokenizer(
        headline,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        pred_1d, pred_2d, pred_3d = model(input_ids, attention_mask)
    
    return {
        'headline': headline,
        'ret_1d': float(pred_1d.squeeze(-1).cpu().numpy()[0]),
        'ret_2d': float(pred_2d.squeeze(-1).cpu().numpy()[0]),
        'ret_3d': float(pred_3d.squeeze(-1).cpu().numpy()[0]),
    }

# Test predictions on sample headlines
print("Making Predictions on Sample Headlines:")
print("="*70)

sample_headlines = [
    "Company reports better than expected quarterly earnings",
    "Stock price drops on negative news about market conditions",
    "New product launch expected to boost sales and revenue",
    "RBI hikes interest rates affecting banking sector negatively",
    "Company announces merger creating strong synergies"
]

predictions_list = []
for headline in sample_headlines:
    pred = predict_single_headline(headline, model, tokenizer, device)
    predictions_list.append(pred)
    print(f"\nHeadline: {headline[:60]}...")
    print(f"  1-Day Return Prediction: {pred['ret_1d']:+.6f}")
    print(f"  2-Day Return Prediction: {pred['ret_2d']:+.6f}")
    print(f"  3-Day Return Prediction: {pred['ret_3d']:+.6f}")

print("\n" + "="*70)

In [ ]:
# Save model and results
print("\nSaving Model and Results...")
print("="*70)

# Save model weights
torch.save(model.state_dict(), 'trained_model_weights.pth')
print("✓ Model weights saved: trained_model_weights.pth")

# Save test predictions as CSV
test_results_df = pd.DataFrame({
    'true_ret_1d': test_actuals['1d'],
    'pred_ret_1d': test_preds['1d'],
    'true_ret_2d': test_actuals['2d'],
    'pred_ret_2d': test_preds['2d'],
    'true_ret_3d': test_actuals['3d'],
    'pred_ret_3d': test_preds['3d'],
})

test_results_df.to_csv('test_predictions_results.csv', index=False)
print("✓ Test predictions saved: test_predictions_results.csv")

# Save model metrics as JSON
import json
metrics_dict = {}
for key, value in test_metrics.items():
    metrics_dict[key] = {k: float(v) for k, v in value.items()}

with open('model_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print("✓ Model metrics saved: model_metrics.json")

print("\n" + "="*70)
print("✓ Model Training Complete!")
print("="*70)

print("\nModel Summary:")
print(f"  Total samples used: {len(df_clean)}")
print(f"  Training samples: {len(train_df)}")
print(f"  Validation samples: {len(val_df)}")
print(f"  Test samples: {len(test_df)}")
print(f"  Model: BERT-base-uncased with Multi-Task Learning heads")
print(f"  Best validation loss: {best_val_loss:.6f}")

print("\nKey Results:")
for horizon, metrics in test_metrics.items():
    print(f"\n  {horizon.upper()} Prediction:")
    print(f"    MSE:  {metrics['MSE']:.6f}")
    print(f"    MAE:  {metrics['MAE']:.6f}")
    print(f"    RMSE: {metrics['RMSE']:.6f}")
    print(f"    R²:   {metrics['R2']:.6f}")